# Generation run (Colab)
Use an **L4 or A100** runtime (a T4 works but sampling is ~20x slower). Run cells 1-3 at the start of every session, then continue with the first unfinished step. Every step is resumable: re-running it skips finished questions.

**To stop a running cell, use the stop button** (not Disconnect) so the current file is closed and saved to Drive.

In [ ]:
# 1. Mount Drive and get/update the code (outputs live in Drive next to it)
from google.colab import drive; drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/coms4705-hallucination-gen'
REPO_URL = 'https://github.com/nataliecclaire/NLP_final_project.git'
if not os.path.exists(PROJ):
    !git clone {REPO_URL} {PROJ}
%cd {PROJ}
!git pull --ff-only
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install
!pip install -q -r requirements.txt

In [ ]:
# 3. (Optional) Hugging Face login with a Read token - the model is not gated; this just avoids download rate limits
from huggingface_hub import login; login()

## Step A: 50 TriviaQA questions end to end (done)

In [ ]:
!python scripts/generate.py --dataset triviaqa --split train --limit 50

In [ ]:
!python -m pytest -s

## Step B: full run
### B1. Greedy answers (HF transformers, one question at a time)
GSM8K is the slow part (~20 s/question on an L4, ~3 h for all 500).

In [ ]:
!python scripts/generate.py --dataset all --split all --phase greedy

### B2. Install vLLM for fast sampling
If this fails, skip to B4: it falls back to HF transformers (much slower).

In [ ]:
!pip install -q vllm
!python -c "import vllm; print('vLLM', vllm.__version__)"

### B3. Check vLLM against the saved HF greedy outputs
Must end with `vLLM CHECK PASSED` before running B4 with vLLM.

In [ ]:
!python scripts/check_vllm.py

### B4. 10 samples per question

In [ ]:
!python scripts/generate.py --dataset all --split all --phase sample

### B5. Merge into the final files + summary

In [ ]:
!python scripts/generate.py --dataset all --split all --phase merge

### B6. Make sure everything is written to Drive before closing the session

In [ ]:
from google.colab import drive; drive.flush_and_unmount()